In [13]:
import pandas as pd

df_adult = pd.read_csv("data/raw/adult.csv")

print("ADULT INCOME")
print("Shape:", df_adult.shape)
print("\nTipos de datos:\n", df_adult.dtypes)
print("\nHead:\n", df_adult.head())
print("\nTail:\n", df_adult.tail())
assert not df_adult.empty, "El dataset de Adult está vacío."

ADULT INCOME
Shape: (32561, 15)

Tipos de datos:
 0     int64
1       str
2     int64
3       str
4     int64
5       str
6       str
7       str
8       str
9       str
10    int64
11    int64
12    int64
13      str
14      str
dtype: object

Head:
     0                  1       2           3   4                    5  \
0  39          State-gov   77516   Bachelors  13        Never-married   
1  50   Self-emp-not-inc   83311   Bachelors  13   Married-civ-spouse   
2  38            Private  215646     HS-grad   9             Divorced   
3  53            Private  234721        11th   7   Married-civ-spouse   
4  28            Private  338409   Bachelors  13   Married-civ-spouse   

                    6               7       8        9    10  11  12  \
0        Adm-clerical   Not-in-family   White     Male  2174   0  40   
1     Exec-managerial         Husband   White     Male     0   0  13   
2   Handlers-cleaners   Not-in-family   White     Male     0   0  40   
3   Handlers-cleaners

In [14]:
import pandas as pd


df_student = pd.read_csv("data/raw/student.csv")

print("STUDENT DROPOUT")
print("Dimensiones (Shape):", df_student.shape)
print("\nTipos de datos (Dtypes):\n", df_student.dtypes)
print("\nPrimeras filas (Head):\n", df_student.head())
print("\nÚltimas filas (Tail):\n", df_student.tail())
assert not df_student.empty, "El dataset de Student está vacío"

STUDENT DROPOUT
Dimensiones (Shape): (4424, 37)

Tipos de datos (Dtypes):
 Marital Status                                      int64
Application mode                                    int64
Application order                                   int64
Course                                              int64
Daytime/evening attendance                          int64
Previous qualification                              int64
Previous qualification (grade)                    float64
Nacionality                                         int64
Mother's qualification                              int64
Father's qualification                              int64
Mother's occupation                                 int64
Father's occupation                                 int64
Admission grade                                   float64
Displaced                                           int64
Educational special needs                           int64
Debtor                                              int

In [15]:
import pandas as pd

# Adult Income
df_adult = pd.read_csv("data/raw/adult.csv")
audit_adult = pd.DataFrame({
    "tipo": df_adult.dtypes.astype(str),
    "ausentes": df_adult.isna().sum(),
    "porcentaje_ausente": (df_adult.isna().mean() * 100).round(2),
    "unicos": df_adult.nunique(dropna=False)
}).sort_values("porcentaje_ausente", ascending=False)

print("Adult Income:")
print("Filas duplicadas:", df_adult.duplicated().sum())
print(audit_adult)

# Student Dropout
df_student = pd.read_csv("data/raw/student.csv")
audit_student = pd.DataFrame({
    "tipo": df_student.dtypes.astype(str),
    "ausentes": df_student.isna().sum(),
    "porcentaje_ausente": (df_student.isna().mean() * 100).round(2),
    "unicos": df_student.nunique(dropna=False)
}).sort_values("porcentaje_ausente", ascending=False)

print("\nStudent Dropout:")
print("Filas duplicadas:", df_student.duplicated().sum())
print(audit_student)


Adult Income:
Filas duplicadas: 24
     tipo  ausentes  porcentaje_ausente  unicos
0   int64         0                 0.0      73
1     str         0                 0.0       9
2   int64         0                 0.0   21648
3     str         0                 0.0      16
4   int64         0                 0.0      16
5     str         0                 0.0       7
6     str         0                 0.0      15
7     str         0                 0.0       6
8     str         0                 0.0       5
9     str         0                 0.0       2
10  int64         0                 0.0     119
11  int64         0                 0.0      92
12  int64         0                 0.0      94
13    str         0                 0.0      42
14    str         0                 0.0       2

Student Dropout:
Filas duplicadas: 0
                                                   tipo  ausentes  \
Marital Status                                    int64         0   
Application mode     

In [16]:
import pandas as pd

# Adult Income 
columns_adult = [
    "age","workclass","fnlwgt","education","education_num","marital_status",
    "occupation","relationship","race","sex","capital_gain","capital_loss",
    "hours_per_week","native_country","income"
]

df_adult = pd.read_csv("data/raw/adult.csv", names=columns_adult)

TARGET_ADULT = "income"
DROP_COLUMNS_ADULT = []  

y_adult = df_adult[TARGET_ADULT]
X_adult = df_adult.drop(columns=[TARGET_ADULT] + DROP_COLUMNS_ADULT)

print("Adult Income - Distribución del target:")
print(y_adult.value_counts(dropna=False))
assert y_adult.notna().all(), "El target de Adult Income tiene valores nulos"
assert y_adult.nunique() >= 2, "El target de Adult Income debe tener al menos dos clases"

# Student Dropout
df_student = pd.read_csv("data/raw/student.csv")

TARGET_STUDENT = "Target"
DROP_COLUMNS_STUDENT = [] 

y_student = df_student[TARGET_STUDENT]
X_student = df_student.drop(columns=[TARGET_STUDENT] + DROP_COLUMNS_STUDENT)

print("\nStudent Dropout - Distribución del target:")
print(y_student.value_counts(dropna=False))
assert y_student.notna().all(), "El target de Student Dropout tiene valores nulos"
assert y_student.nunique() >= 2, "El target de Student Dropout debe tener al menos dos clases"


Adult Income - Distribución del target:
income
 <=50K    24720
 >50K      7841
14            1
Name: count, dtype: int64

Student Dropout - Distribución del target:
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64


In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Adult Income 
num_cols_adult = X_adult.select_dtypes(include="number").columns.tolist()
cat_cols_adult = X_adult.select_dtypes(exclude="number").columns.tolist()

num_pipe_adult = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

cat_pipe_adult = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess_adult = ColumnTransformer([
    ("num", num_pipe_adult, num_cols_adult),
    ("cat", cat_pipe_adult, cat_cols_adult)
])

print("Adult Income - Numéricas:", len(num_cols_adult), "Categóricas:", len(cat_cols_adult))

# Student Dropout
num_cols_student = X_student.select_dtypes(include="number").columns.tolist()
cat_cols_student = X_student.select_dtypes(exclude="number").columns.tolist()

num_pipe_student = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

cat_pipe_student = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess_student = ColumnTransformer([
    ("num", num_pipe_student, num_cols_student),
    ("cat", cat_pipe_student, cat_cols_student)
])

print("Student Dropout - Numéricas:", len(num_cols_student), "Categóricas:", len(cat_cols_student))


Adult Income - Numéricas: 6 Categóricas: 8
Student Dropout - Numéricas: 36 Categóricas: 0


In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score
from sklearn.pipeline import Pipeline

# Adult Income
columns_adult = [
    "age","workclass","fnlwgt","education","education_num","marital_status",
    "occupation","relationship","race","sex","capital_gain","capital_loss",
    "hours_per_week","native_country","income"
]

df_adult = pd.read_csv("data/raw/adult.csv", names=columns_adult)

df_adult["income"] = df_adult["income"].str.strip()

df_adult = df_adult[df_adult["income"].isin(["<=50K", ">50K"])]

print("Adult Income - filas después del filtrado:", len(df_adult))
assert len(df_adult) > 0, "Adult Income quedó vacío después del filtrado"

y_adult = df_adult["income"]
X_adult = df_adult.drop(columns=["income"])

Xtr_a, Xte_a, ytr_a, yte_a = train_test_split(
    X_adult, y_adult, test_size=0.20, random_state=42, stratify=y_adult
)

dummy_adult = Pipeline([
    ("prep", preprocess_adult),
    ("model", DummyClassifier(strategy="most_frequent"))
])

svm_adult = Pipeline([
    ("prep", preprocess_adult),
    ("model", SVC(C=1, gamma="scale", probability=True))
])

print("\nAdult Income:")
for name, model in {"dummy": dummy_adult, "svm": svm_adult}.items():
    model.fit(Xtr_a, ytr_a)
    pred = model.predict(Xte_a)
    print(name, f1_score(yte_a, pred, average="macro"))
print(classification_report(yte_a, svm_adult.predict(Xte_a)))

#  Student Dropout 
df_student = pd.read_csv("data/raw/student.csv")

y_student = df_student["Target"]
X_student = df_student.drop(columns=["Target"])

Xtr_s, Xte_s, ytr_s, yte_s = train_test_split(
    X_student, y_student, test_size=0.20, random_state=42, stratify=y_student
)

dummy_student = Pipeline([
    ("prep", preprocess_student),
    ("model", DummyClassifier(strategy="most_frequent"))
])

svm_student = Pipeline([
    ("prep", preprocess_student),
    ("model", SVC(C=1, gamma="scale", probability=True))
])

print("\nStudent Dropout:")
for name, model in {"dummy": dummy_student, "svm": svm_student}.items():
    model.fit(Xtr_s, ytr_s)
    pred = model.predict(Xte_s)
    print(name, f1_score(yte_s, pred, average="macro"))
print(classification_report(yte_s, svm_student.predict(Xte_s)))


Adult Income - filas después del filtrado: 32561

Adult Income:
dummy 0.4315761913073835


c:\Users\marli\INF8239_U01\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


svm 0.7926166860412465
              precision    recall  f1-score   support

       <=50K       0.88      0.94      0.91      4945
        >50K       0.77      0.60      0.67      1568

    accuracy                           0.86      6513
   macro avg       0.82      0.77      0.79      6513
weighted avg       0.85      0.86      0.85      6513


Student Dropout:
dummy 0.22205476011052497


c:\Users\marli\INF8239_U01\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


svm 0.6784703597430611
              precision    recall  f1-score   support

     Dropout       0.84      0.73      0.78       284
    Enrolled       0.50      0.35      0.41       159
    Graduate       0.77      0.93      0.84       442

    accuracy                           0.76       885
   macro avg       0.71      0.67      0.68       885
weighted avg       0.75      0.76      0.75       885

